In [ ]:
# Repository setup for portable, repo-relative paths
from pathlib import Path
import sys

def _find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from dx_chat_entropy.paths import get_paths
PATHS = get_paths(REPO_ROOT)


# Differential LR Estimation

Estimate pairwise **differential likelihood ratios** for clinical findings
that discriminate between two diagnosis categories (Disease A vs Disease B).

Input workbook layout:
- Row 0: sparse category labels (exactly 2 categories, forward-filled)
- Row 1: exemplar diseases for each category
- Rows >= 2: clinical findings (column 0)

Two API variants are provided:
1. **Chat Completions API** — the original approach
2. **Responses API** — current approach for GPT-5 / o-series models

In [ ]:
from __future__ import annotations

import os
import pandas as pd
import numpy as np
from pathlib import Path
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — manifest-driven workbook selection
# ──────────────────────────────────────────────────────────────────────────────
# Preferred mode: generated pair manifest from scripts/build_differential_inputs.py.
MANIFEST_PATH = PATHS.processed / "lr_differential" / "manifests" / "pairs_manifest.csv"
SCENARIO_FILTER = None   # e.g. ["chest_pain_carter_8", "metrorrhagia_sperle_tier3"]
MAX_PAIRS = None         # e.g. 20 for chunked runs

# Fallback manual mode (used only if MANIFEST_PATH is missing):
WORKBOOKS = []  # [(input_path, output_path), ...]


## Discriminitave LRs 

#### General, differential likelihood ratio estimators

This answers the question: in general, what are the features that differentiate between Disease A and Disease B. (not specific to a particular case)

Inputs: Correct diagnosis, learner’s differential, and transcript.

Outputs: A list of key evidence that differentiates cases where the learner’s differential diagnosis was correct vs. the actual correct diagnosis.

Challenges:
- Learners may list many possible diagnoses, requiring feedback across a broad range.
- Usefulness is measured by differential LR (A vs. B) rather than the usual (overall) LR (A vs. not A).

Approach:
- Collected all DDx from Cory's list; in production, this can be auto-extracted from the transcript.
- Used GPT-4o to identify key discriminating factors in:
	- HPI, Context (Medical, Surgical, Medications, Social, Family), Vitals/Exam, and Testing.
- In production, we’d automate detecting whether the learner asked about these factors and generate feedback:
	- “X was important, and you asked it.”
	- “Y was important, but you didn’t ask it.”

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Differential-LR estimation for every finding row
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel

# 1.  Prompt helpers 

def generate_diff_gen_prompt(
    dx_cat_1: str,
    dx_cat_2: str,
    dx_cat_1_examples: list[str],
    dx_cat_2_examples: list[str],
) -> str:
    """
    Build a structured prompt for an LLM to estimate the *differential*
    likelihood ratio between two diagnosis categories.  Each category
    is augmented with illustrative exemplar diseases to reinforce the
    desired clinical context (few‑shot cueing).
    """
    # helper to format the exemplar list as comma‑separated text
    def fmt_examples(ex_list: list[str]) -> str:
        return ", ".join(ex_list) if ex_list else "—"

    ex1 = fmt_examples(dx_cat_1_examples)
    ex2 = fmt_examples(dx_cat_2_examples)

    return f"""
Your task: Estimate the **likelihood ratio (LR)** for a clinical finding
that discriminates **{dx_cat_1}** from **{dx_cat_2}** in an outpatient
chest‑pain evaluation.  Assume the patient has **exactly one** of these
two conditions; other diagnoses can be ignored.

**Category examples**  
• {dx_cat_1}: {ex1}  
• {dx_cat_2}: {ex2}

### Working definition
LR = P(finding | {dx_cat_1}) ÷ P(finding | {dx_cat_2})

### Qualitative bands (reference only)
* LR > 10 strong evidence for {dx_cat_1}  
* 5 < LR ≤ 10 moderate evidence for {dx_cat_1}  
* 2 < LR ≤ 5 weak evidence for {dx_cat_1}  
* 0.5 < LR ≤ 2 negligible evidence  
* 0.2 < LR ≤ 0.5 weak evidence for {dx_cat_2}  
* 0.1 < LR ≤ 0.2 moderate evidence for {dx_cat_2}  
* LR ≤ 0.1 strong evidence for {dx_cat_2}

### Response format (JSON, one line)	
{{“lr”: , “strength”: “”, “rationale”: “<≤40 words>”}}
Only output the JSON block—no extra text.
"""


def load_dx_categories(path: str):
    """
    Parse the workbook where row‑0 contains sparse category labels and
    row‑1 lists exemplar diseases.

    Returns
    -------
    dx_cat_1, dx_cat_2       : str
    dx_cat_1_examples        : list[str]
    dx_cat_2_examples        : list[str]
    """
    # read raw without header inference
    df_raw = pd.read_excel(Path(path), header=None)

    # first row → categories; forward‑fill missing cells
    cats = df_raw.iloc[0].ffill()

    # second row → exemplar diseases (may contain NaN)
    ex_row = df_raw.iloc[1]

    # identify distinct category names in order of appearance
    unique_cats = pd.unique(cats.dropna())
    if len(unique_cats) != 2:
        raise ValueError("Expected exactly two category labels in first row")

    dx_cat_1, dx_cat_2 = unique_cats.tolist()

    # build example lists aligned with column categories
    dx_cat_1_examples = ex_row[cats == dx_cat_1].dropna().astype(str).tolist()
    dx_cat_2_examples = ex_row[cats == dx_cat_2].dropna().astype(str).tolist()

    return dx_cat_1, dx_cat_2, dx_cat_1_examples, dx_cat_2_examples

# ---------- response schema -------------------------------------------------
class DiffLR(BaseModel):
    lr: float
    strength: str
    rationale: str

# ---------- single LLM call -------------------------------------------------
def estimate_diff_lr(
    finding: str,
    dx_cat_1: str,
    dx_cat_2: str,
    ex1: list[str],
    ex2: list[str],
    model: str = "gpt-4.1-nano-2025-04-14",
    reasoning_effort: str = "medium",
    client: OpenAI | None = None,
) -> float | str:
    if client is None:
        client = OpenAI()

    prompt = generate_diff_gen_prompt(dx_cat_1, dx_cat_2, ex1, ex2)

    sys_msg  = {"role": "system", "content": prompt}
    user_msg = {"role": "user",  "content": f"Finding: {finding}"}

    kwargs = {}
    if model.startswith("o"):
        kwargs["reasoning_effort"] = reasoning_effort

    try:
        resp = client.beta.chat.completions.parse(
            model=model,
            messages=[sys_msg, user_msg],
            response_format=DiffLR,
            **kwargs,
        )
        return resp.choices[0].message.parsed.lr
    except Exception as exc:
        print(f"LLM error for finding '{finding}': {exc}")
        return "ERROR"

# 2.  File paths --------------------------------------------------------------
#WB_IN  = REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia.xlsx'
#WB_OUT = REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia_filled.xlsx'
WB_IN = REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia.xlsx'
WB_OUT = REPO_ROOT / 'archive/legacy_runs/lr_estimation_2025_07_21/LRs for 87 features within GI ddx - dysphagia vs esophageal pain without dysphagia_filled.xlsx'
MODEL_ID = "gpt-4.1-nano-2025-04-14"

# 3.  Load categories & examples ---------------------------------------------
dx_cat_1, dx_cat_2, ex1, ex2 = load_dx_categories(WB_IN)

# 4.  Read sheet and append results column -----------------------------------
df = pd.read_excel(WB_IN, sheet_name=0, header=None)
df_out = df.copy()

result_col = df_out.shape[1]      # index for new column
df_out[result_col] = pd.array([np.nan] * len(df_out), dtype=object)       # create the column

header_label = (
    f"Differential LR ({MODEL_ID} for '{dx_cat_1}' vs. '{dx_cat_2}')"
)
df_out.iat[0, result_col] = header_label       # descriptive header
df_out.iat[1, result_col] = ""                 # keep exemplar row blank


# 5.  Iterate over findings (rows ≥ 2) ---------------------------------------
client = OpenAI()
total_rows = df_out.shape[0] - 2           # rows of actual findings
done = 0

for row in range(2, df_out.shape[0]):
    finding = str(df_out.iat[row, 0]).strip()
    if not finding:
        continue

    done += 1
    print(f"[{done}/{total_rows}] Estimating LR for: {finding[:60]}")

    lr_val = estimate_diff_lr(
        finding=finding,
        dx_cat_1=dx_cat_1,
        dx_cat_2=dx_cat_2,
        ex1=ex1,
        ex2=ex2,
        model=MODEL_ID,
        reasoning_effort="medium",
        client=client,
    )
    df_out.iat[row, result_col] = lr_val

# 6.  Save workbook -----------------------------------------------------------
df_out.to_excel(WB_OUT, index=False, header=False)
print(f"Saved filled sheet to {WB_OUT}")

New version for GPT-5 interface. Note, might be fine to just use low (rather than medium) reasoning. 

Note that this requires some set up: needs separate notebooks that have: 
- row 1 contains the categories, spaced such that examples are in the next row and continue until the next category
- note, only allows 2 categories for now - if needed, coudl generalize this.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Differential-LR estimation — Responses API (current approach)
# ─────────────────────────────────────────────────────────────────────────────
# By default this cell reads pair workbooks from MANIFEST_PATH.
# It also supports resumable execution: if an output workbook already exists,
# rows with an existing LR value in the current result column are skipped.
# ─────────────────────────────────────────────────────────────────────────────

def generate_diff_gen_prompt(
    dx_cat_1: str,
    dx_cat_2: str,
    dx_cat_1_examples: list[str],
    dx_cat_2_examples: list[str],
) -> str:
    """Concise prompt; schema enforcement is via Structured Outputs."""

    def fmt_examples(ex_list: list[str]) -> str:
        return ", ".join(ex_list) if ex_list else "-"

    ex1 = fmt_examples(dx_cat_1_examples)
    ex2 = fmt_examples(dx_cat_2_examples)

    return f"""
You are a clinical epidemiologist.

Task: Estimate the *differential* likelihood ratio (LR) for a single clinical finding
that discriminates **{dx_cat_1}** from **{dx_cat_2}** in an outpatient diagnostic workup.
Assume the patient has exactly one of these two conditions.

Category examples:
• {dx_cat_1}: {ex1}
• {dx_cat_2}: {ex2}

Definition
LR = P(finding | {dx_cat_1}) / P(finding | {dx_cat_2})

Reference bands (for interpreting strength):
>10 strong for {dx_cat_1}; 5-10 moderate; 2-5 weak; 0.5-2 negligible;
0.2-0.5 weak for {dx_cat_2}; 0.1-0.2 moderate; <=0.1 strong for {dx_cat_2}.
""".strip()


def load_dx_categories(path: str | Path):
    df_raw = pd.read_excel(Path(path), header=None)
    cats = df_raw.iloc[0].ffill()
    ex_row = df_raw.iloc[1]

    unique_cats = pd.unique(cats.dropna())
    if len(unique_cats) != 2:
        raise ValueError("Expected exactly two category labels in row 0.")

    dx_cat_1, dx_cat_2 = unique_cats.tolist()
    ex1 = ex_row[cats == dx_cat_1].dropna().astype(str).tolist()
    ex2 = ex_row[cats == dx_cat_2].dropna().astype(str).tolist()
    return dx_cat_1, dx_cat_2, ex1, ex2


class DiffLR(BaseModel):
    lr: float
    strength: str | None = None
    rationale: str | None = None


MODEL_ID = "gpt-4.1-nano-2025-04-14"
REASONING_EFFORT = "medium"   # minimal | low | medium | high
VERBOSITY = "medium"           # gpt-4.1-nano only supports "medium"

client = OpenAI()


def estimate_diff_lr(
    finding: str,
    dx_cat_1: str,
    dx_cat_2: str,
    ex1: list[str],
    ex2: list[str],
) -> float | str:
    prompt = generate_diff_gen_prompt(dx_cat_1, dx_cat_2, ex1, ex2)
    kwargs = {}
    if MODEL_ID.startswith("o"):
        kwargs["reasoning"] = {"effort": REASONING_EFFORT}
    try:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Finding: {finding}"},
            ],
            text_format=DiffLR,
            text={"verbosity": VERBOSITY},
            **kwargs,
        )
        return float(resp.output_parsed.lr)
    except Exception as exc:
        print(f"LLM error for finding '{finding}': {exc}")
        return "ERROR"


def _normalize_scenario_filter(value):
    if value is None:
        return None
    if isinstance(value, str):
        return {value}
    return {str(item) for item in value}


def load_workbooks_from_manifest(
    manifest_path: str | Path,
    scenario_filter=None,
    max_pairs: int | None = None,
):
    manifest = Path(manifest_path)
    if not manifest.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest}")

    df_manifest = pd.read_csv(manifest)
    required = {"input_workbook", "output_workbook"}
    missing = sorted(required - set(df_manifest.columns))
    if missing:
        raise ValueError(f"Manifest missing required columns: {missing}")

    if "scenario_id" not in df_manifest.columns:
        df_manifest["scenario_id"] = "unknown"
    if "pair_index" not in df_manifest.columns:
        df_manifest["pair_index"] = range(1, len(df_manifest) + 1)

    selected = df_manifest.sort_values(["scenario_id", "pair_index"]).copy()

    allowed = _normalize_scenario_filter(scenario_filter)
    if allowed is not None:
        selected = selected[selected["scenario_id"].isin(allowed)]

    if max_pairs is not None:
        selected = selected.head(int(max_pairs))

    workbooks = []
    for row in selected.itertuples(index=False):
        wb_in = REPO_ROOT / str(row.input_workbook)
        wb_out = REPO_ROOT / str(row.output_workbook)
        workbooks.append((wb_in, wb_out, str(row.scenario_id)))
    return workbooks


if Path(MANIFEST_PATH).exists():
    workbook_records = load_workbooks_from_manifest(
        MANIFEST_PATH,
        scenario_filter=SCENARIO_FILTER,
        max_pairs=MAX_PAIRS,
    )
else:
    workbook_records = [(Path(wb_in), Path(wb_out), "manual") for wb_in, wb_out in WORKBOOKS]

if not workbook_records:
    raise ValueError("No workbooks selected. Check MANIFEST_PATH, SCENARIO_FILTER, and MAX_PAIRS.")

print(f"Selected {len(workbook_records)} workbook pair(s) for estimation.")


# ─── Process selected workbooks ──────────────────────────────────────────────
for wb_in, wb_out, scenario_id in workbook_records:
    print(f"\n{'=' * 80}")
    print(f"Scenario: {scenario_id}")
    print(f"Input:  {wb_in}")
    print(f"Output: {wb_out}")
    print(f"{'=' * 80}")

    if not wb_in.exists():
        print(f"Skipping missing input workbook: {wb_in}")
        continue

    dx_cat_1, dx_cat_2, ex1, ex2 = load_dx_categories(wb_in)

    df_in = pd.read_excel(wb_in, sheet_name=0, header=None)
    if wb_out.exists():
        df_out = pd.read_excel(wb_out, sheet_name=0, header=None)
        if df_out.shape[0] < df_in.shape[0]:
            df_out = df_out.reindex(range(df_in.shape[0]))
    else:
        df_out = df_in.copy()

    if df_out.shape[1] < df_in.shape[1]:
        for _ in range(df_in.shape[1] - df_out.shape[1]):
            df_out[df_out.shape[1]] = None

    # Ensure source columns mirror current input workbook.
    for col in range(df_in.shape[1]):
        df_out[col] = df_in[col]

    header_label = f"Differential LR ({MODEL_ID} for '{dx_cat_1}' vs. '{dx_cat_2}')"

    result_col = None
    for col in range(df_out.shape[1]):
        if str(df_out.iat[0, col]).strip() == header_label:
            result_col = col
            break
    if result_col is None:
        result_col = df_out.shape[1]
        df_out[result_col] = pd.Series([None] * len(df_out), dtype="object")

    df_out.iat[0, result_col] = header_label
    if len(df_out) > 1:
        df_out.iat[1, result_col] = ""

    finding_rows = []
    for row_idx in range(2, df_in.shape[0]):
        finding = str(df_in.iat[row_idx, 0]).strip()
        if finding:
            finding_rows.append(row_idx)

    pending_rows = []
    for row_idx in finding_rows:
        existing = df_out.iat[row_idx, result_col] if row_idx < df_out.shape[0] else None
        if pd.isna(existing) or str(existing).strip() == "":
            pending_rows.append(row_idx)

    print(
        f"Rows with findings: {len(finding_rows)} | "
        f"Already filled: {len(finding_rows) - len(pending_rows)} | "
        f"Pending: {len(pending_rows)}"
    )

    for idx, row_idx in enumerate(pending_rows, start=1):
        finding = str(df_in.iat[row_idx, 0]).strip()
        print(f"[{idx}/{len(pending_rows)}] Estimating LR for: {finding[:80]}")

        lr_val = estimate_diff_lr(
            finding=finding,
            dx_cat_1=dx_cat_1,
            dx_cat_2=dx_cat_2,
            ex1=ex1,
            ex2=ex2,
        )
        df_out.iat[row_idx, result_col] = lr_val

    wb_out.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_excel(wb_out, index=False, header=False)
    print(f"Saved -> {wb_out}")
